# 01 — Data Understanding

## Objective
Understand the structure, quality, and business meaning of the Home Credit
Default Risk dataset before any cleaning or modeling begins.

## Dataset
- Source: Home Credit Default Risk (Kaggle competition)
- File used: `application_train.csv`
- Shape: 307,511 rows × 122 columns
- Target: `TARGET` (1 = defaulted, 0 = repaid)

In [ ]:
import pandas as pd

train = pd.read_csv('../data/raw/application_train.csv')
print(train.shape)
print(f"{train.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

In [ ]:
train.info()

In [ ]:
train['TARGET'].value_counts()

In [ ]:
train['TARGET'].value_counts(normalize=True)

### Finding: Class imbalance in TARGET

**Observation:**
~91.9% of applicants did not default (TARGET=0), ~8.1% did (TARGET=1).

**Investigation:**
Ran `train['TARGET'].value_counts(normalize=True)`.

**Conclusion:**
This dataset only contains already-approved applicants — Home Credit's own
underwriting process already filtered out obviously high-risk applicants
before this data was collected. This is a form of sample selection bias:
the 8% default rate reflects risk *among approved applicants*, not the
general applicant population.

**Implication for later steps:**
- Accuracy will be a misleading evaluation metric (a model predicting
  "no default" for everyone would already score ~92%).
- Will need to prioritize Recall/Precision/ROC-AUC/F1 in model evaluation
  (notebook 05).
- Will note this limitation explicitly in the README's model limitations
  section — the model predicts risk among pre-approved applicants, not
  the full applicant pool.

In [ ]:
train.dtypes.value_counts()

In [ ]:
categorical_cols = train.select_dtypes(include='str').columns.tolist()
print(categorical_cols)

In [ ]:
for col in ['CODE_GENDER', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'OCCUPATION_TYPE']:
    print(f"--- {col} ---")
    print(train[col].value_counts(dropna=False))
    print()

In [30]:
# Check if missing OCCUPATION_TYPE lines up with income type (employment status)
train[train['OCCUPATION_TYPE'].isna()]['NAME_INCOME_TYPE'].value_counts()

NAME_INCOME_TYPE
Pensioner               55357
Working                 24920
Commercial associate    12297
State servant            3787
Unemployed                 22
Student                     5
Businessman                 2
Maternity leave             1
Name: count, dtype: int64

In [ ]:
train['NAME_INCOME_TYPE'].value_counts()

### Finding: Missing values in OCCUPATION_TYPE

**Observation:**
96,391 of 307,511 rows (~31%) have missing OCCUPATION_TYPE.

**Investigation:**
Cross-tabulated missing OCCUPATION_TYPE rows against NAME_INCOME_TYPE.
Found: ~100% of Pensioners and Unemployed applicants are missing occupation,
vs. only ~15.7% of "Working" applicants.

**Conclusion:**
Missingness is largely structural — Pensioners/Unemployed have no current
occupation to report, which explains the vast majority of the gap. A smaller,
secondary ~16% missingness rate among currently "Working" applicants remains
unexplained — likely a genuine data collection gap rather than a structural
pattern.

**Implication for later steps:**
- Create explicit "Not Employed" category for Pensioners/Unemployed during
  preprocessing (informative missingness, not noise).
- Handle residual "Working" gap separately, likely as generic "Unknown"
  category, since no clean structural cause was found.

In [28]:
train['ORGANIZATION_TYPE'].value_counts(dropna=False)

ORGANIZATION_TYPE
Business Entity Type 3    67992
XNA                       55374
Self-employed             38412
Other                     16683
Medicine                  11193
Business Entity Type 2    10553
Government                10404
School                     8893
Trade: type 7              7831
Kindergarten               6880
Construction               6721
Business Entity Type 1     5984
Transport: type 4          5398
Trade: type 3              3492
Industry: type 9           3368
Industry: type 3           3278
Security                   3247
Housing                    2958
Industry: type 11          2704
Military                   2634
Bank                       2507
Agriculture                2454
Police                     2341
Transport: type 2          2204
Postal                     2157
Security Ministries        1974
Trade: type 2              1900
Restaurant                 1811
Services                   1575
University                 1327
Industry: type 7      

In [ ]:
train['NAME_HOUSING_TYPE'].value_counts(dropna=False)

In [ ]:
train[train['ORGANIZATION_TYPE'] == 'XNA']['NAME_INCOME_TYPE'].value_counts()